🔵 1️⃣ Legacy mode (vanilla CLASS behavior) 


This behaves exactly like standard CLASS.
No split keys used.

In [ ]:
import matplotlib as plt   

In [ ]:
from classy_NEDE import Class

params_legacy = {
    # Basic cosmology
    "output": "mPk",
    "H0": 67.5,
    "omega_b": 0.022,
    "omega_cdm": 0.12,
    "tau_reio": 0.054,
    "n_s": 0.965,
    "ln10^{10}A_s": 3.0,

    # One massive neutrino species (degeneracy = 3)
    "N_ncdm": 1,
    "m_ncdm": "0.1",
    "deg_ncdm": "3.0",

    # No interactions
    "G_eff_ncdm": 0.0
}

cosmo = Class()
cosmo.set(params_legacy)
cosmo.compute()


What happens internally

- pba->N_ncdm = 1

- pba->N_ncdm_standard = 1

- pba->N_ncdm_interacting = 0

- G_eff_species = [0]

🟢 2️⃣ Mixed sector (standard + interacting)



This is the new feature you implemented.

Example:

1 standard species

2 interacting species

Different couplings per interacting species

In [ ]:
from classy_NEDE import Class

params_mixed = {
    # Basic cosmology
    "output": "mPk",
    "H0": 67.5,
    "omega_b": 0.022,
    "omega_cdm": 0.12,
    "tau_reio": 0.054,
    "n_s": 0.965,
    "ln10^{10}A_s": 3.0,

    # ---- SPLIT MODE ----
    "N_ncdm_standard": 1,
    "N_ncdm_interacting": 2,

    # Provide total lists (length = 3)
    "m_ncdm": "0.1, 0.1, 0.1",
    "deg_ncdm": "1.0, 1.0, 1.0",

    # Different couplings for interacting species
    # (standard species automatically gets 0)
    "G_eff_ncdm_interacting_species": "1e5, 1e9"
}

cosmo = Class()
cosmo.set(params_mixed)
cosmo.compute()


Internally this becomes:
- Nstd = 1
- Nint = 2
- Ntot = 3


G_eff_species = [0, 1e3, 1e5]


So yes — you now fully support:

ppt->G_eff_ncdm_species = [0, 0, G1, G2, ..., Gn]

🔴 3️⃣ Fully interacting sector (all species interacting)

This reproduces Tobias-style physics but using your split machinery.

In [ ]:
from classy_NEDE import Class

params_full_interacting = {
    # Basic cosmology
    "output": "mPk",
    "H0": 67.5,
    "omega_b": 0.022,
    "omega_cdm": 0.12,
    "tau_reio": 0.054,
    "n_s": 0.965,
    "ln10^{10}A_s": 3.0,

    # ---- SPLIT MODE ----
    "N_ncdm_standard": 0,
    "N_ncdm_interacting": 3,

    "m_ncdm": "0.1, 0.1, 0.1",
    "deg_ncdm": "1.0, 1.0, 1.0",

    # Broadcast value to all interacting species
    "G_eff_ncdm_interacting": "1e6"
}

cosmo = Class()
cosmo.set(params_full_interacting)
cosmo.compute()


Internally:


- Nstd = 0
- Nint = 3
- G_eff_species = [1e6, 1e6, 1e6]


Exactly equivalent to “all neutrinos interacting”.


⚙️ Most General Possible Case

You can also directly set arbitrary couplings for every species:

"G_eff_ncdm_species": "0, 0, 1e3, 1e4, 1e5"


That is the most powerful interface.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy_NEDE import Class

# -----------------------------
# Choose which parameter dicts you defined above
# (must exist in your notebook)
# -----------------------------
MODELS = [
    ("Legacy", params_legacy),
    ("Mixed", params_mixed),
    ("Fully interacting", params_full_interacting),
]

# -----------------------------
# Settings
# -----------------------------
z_plot = 0.0
k_hMpc = np.logspace(-3, 0.0, 400)  # k in h/Mpc (what we plot on the x-axis)

# -----------------------------
# Helper: compute P(k) for a model dict
# -----------------------------
def compute_pk(params, k_hMpc, z=0.0):
    cosmo = Class()
    cosmo.set(params)
    cosmo.compute()

    h = cosmo.h()
    # classy pk(k,z) expects k in 1/Mpc, so we pass k*h (since k_hMpc is in h/Mpc)
    # Convert back to (Mpc/h)^3 by multiplying by h^3
    pk = np.array([cosmo.pk(float(k*h), float(z)) * h**3 for k in k_hMpc])

    # cleanup (important if you run many times in a notebook)
    cosmo.struct_cleanup()
    cosmo.empty()

    return pk

# -----------------------------
# Compute all spectra
# -----------------------------
Pk = {}
for name, params in MODELS:
    print(f"Computing {name}...")
    Pk[name] = compute_pk(params, k_hMpc, z=z_plot)

# -----------------------------
# Plot absolute spectra
# -----------------------------
plt.figure(figsize=(8, 5))
for name, _ in MODELS:
    plt.loglog(k_hMpc, Pk[name], label=name)
plt.xlabel(r"$k\,[h/\mathrm{Mpc}]$")
plt.ylabel(r"$P(k)\,[(\mathrm{Mpc}/h)^3]$")
plt.title(rf"Matter Power Spectrum at $z={z_plot}$")
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()

# -----------------------------
# Plot ratios to Legacy (if present)
# -----------------------------
if "Legacy" in Pk:
    Pk_ref = Pk["Legacy"]
    plt.figure(figsize=(8, 4.5))
    for name, _ in MODELS:
        if name == "Legacy":
            continue
        plt.semilogx(k_hMpc, Pk[name] / Pk_ref, label=f"{name} / Legacy")
    plt.axhline(1.0, lw=1)
    plt.xlabel(r"$k\,[h/\mathrm{Mpc}]$")
    plt.ylabel("Ratio")
    plt.title(rf"Ratios to Legacy at $z={z_plot}$")
    plt.grid(True, which="both", ls=":")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No 'Legacy' model found in MODELS, so ratio plot was skipped.")
